## Загрузка и преобразование датасетов

В данном ноуте осуществляем загрузку и обработку 5 датасетов, применяемых в исследовании. Приводим их к финальному виду для применения моделей и тестирования различных мметодов подбора гиперпараметров.

### Загрузка всех датасетов

In [1]:
!pip install ucimlrepo
!pip install openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.6 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=f444f551573f4b76eb0a690afb5f903ea16632125e7e40bd5722fcb35d3c95e0
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import os

In [3]:
# Словарь с ID датасетов на OpenML
dataset_ids = {
    "Adult": 1590,
    "Bank Marketing": 1461,
    #"Credit Card Clients": 29,
    #"Covertype": 293,
    "Spambase": 44
}

datasets = {}
for name, openml_id in dataset_ids.items():
    print(f"Загружаю датасет: {name} (OpenML ID={openml_id})...")
    try:
        #Загрузка по ID
        X, y = fetch_openml(data_id=openml_id, as_frame=True, parser='auto', return_X_y=True)
        datasets[name] = {'X': X, 'y': y}
        print(f"  -> Готово! Размер: {X.shape}, Целевая переменная: {y.name}")
    except Exception as e:
        print(f"  -> Ошибка при загрузке датасета {name}: {e}. Пробую по имени...")
        # Загрузка по имени
        try:
            X, y = fetch_openml(name=name.lower().replace(' ', '-'), as_frame=True, return_X_y=True)
            datasets[name] = {'X': X, 'y': y}
            print(f"  -> Готово по имени! Размер: {X.shape}")
        except Exception as e2:
            print(f"  -> Не удалось загрузить: {e2}")

print("\nЗагруженные датасеты:", list(datasets.keys()))

Загружаю датасет: Adult (OpenML ID=1590)...
  -> Готово! Размер: (48842, 14), Целевая переменная: class
Загружаю датасет: Bank Marketing (OpenML ID=1461)...
  -> Готово! Размер: (45211, 16), Целевая переменная: Class
Загружаю датасет: Spambase (OpenML ID=44)...
  -> Готово! Размер: (4601, 57), Целевая переменная: class

Загруженные датасеты: ['Adult', 'Bank Marketing', 'Spambase']


## Обработка датасетов

### 1. Adult (Предсказание класса заработка)

**Описание:** В данном датасете представлена информация о жителях США, их различных хараетристиках (14 столбцов). Задача - предсказать, превышает ли годовой доход человека $50000 в год

**Ссылка с описанием:** (https://archive.ics.uci.edu/dataset/2/adult)

#### Обработка Null и косметика

In [4]:
feats_ad = datasets['Adult']['X']
target_ad = datasets['Adult']['y']
feats_ad.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [5]:
target_ad.head()

,class
0,<=50K
1,<=50K
2,>50K
3,>50K
4,<=50K


In [6]:
target_ad.value_counts()

,count
class,
<=50K,37155
>50K,11687


In [7]:
feats_ad = feats_ad.replace(' ?', np.nan)
nans_ad = feats_ad.isna().sum()/len(feats_ad)
nans_ad.head(20)

,0
age,0.000000
workclass,0.057307
fnlwgt,0.000000
education,0.000000
education-num,0.000000
marital-status,0.000000
occupation,0.057512
relationship,0.000000
race,0.000000
sex,0.000000


Видим, что есть Nullы. Заполним их отдельным классом Unknown, так как в данном случае именно отсутствие данных является информацией.

In [8]:
cols_with_nulls = list(nans_ad[nans_ad > 0].index)

for col in cols_with_nulls:
    if pd.api.types.is_categorical_dtype(feats_ad[col]):
        feats_ad[col] = feats_ad[col].cat.add_categories('Unknown')
    feats_ad[col] = feats_ad[col].fillna('Unknown')

/tmp/ipykernel_18687/1083366863.py:4: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(feats_ad[col]):


In [9]:
#Проверяем, что пропусков больше нет
print("Пропуски после обработки:")
print(feats_ad.isna().sum().sum())

for col in cols_with_nulls:
    print(f"\nУникальные значения {col}:")
    print(feats_ad[col].unique())

Пропуски после обработки:
0

Уникальные значения workclass:
['Private', 'Local-gov', 'Unknown', 'Self-emp-not-inc', 'Federal-gov', 'State-gov', 'Self-emp-inc', 'Without-pay', 'Never-worked']
Categories (9, object): ['Federal-gov', 'Local-gov', 'Never-worked', 'Private', ...,
                         'Self-emp-not-inc', 'State-gov', 'Without-pay', 'Unknown']

Уникальные значения occupation:
['Machine-op-inspct', 'Farming-fishing', 'Protective-serv', 'Unknown', 'Other-service', ..., 'Sales', 'Priv-house-serv', 'Transport-moving', 'Handlers-cleaners', 'Armed-Forces']
Length: 15
Categories (15, object): ['Adm-clerical', 'Armed-Forces', 'Craft-repair', 'Exec-managerial', ...,
                          'Sales', 'Tech-support', 'Transport-moving', 'Unknown']

Уникальные значения native-country:
['United-States', 'Unknown', 'Peru', 'Guatemala', 'Mexico', ..., 'Greece', 'Trinadad&Tobago', 'Outlying-US(Guam-USVI-etc)', 'France', 'Holand-Netherlands']
Length: 42
Categories (42, object): ['Cambodi

#### Преобразование таргета

In [10]:
y_base = target_ad.copy()


#Преобразуем таргет
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y_base)

df_adult = feats_ad.copy()
df_adult['target'] = y_encoded

Сохраняем в CSV

In [11]:
output_dir = 'prepared_datasets'
os.makedirs(output_dir, exist_ok=True)
print(f"Датасеты будут сохранены в папку: '{output_dir}/'")
print()

filename = os.path.join(output_dir, 'adult_cleaned_new.csv')
df_adult.to_csv(filename, index=False)

Датасеты будут сохранены в папку: 'prepared_datasets/'



### 2. Bank Marketing (Предсказание открытия депозита)

**Описание:** В данном датасете представлены данные о маркетинговой кампании португальского банка, целью которой было привлечь клиентов к открытию депозитного счета. Задача - предсказать, откроет ли клиент дипозит.

**Ссылка с описанием:** (https://archive.ics.uci.edu/dataset/222/bank+marketing)

Тут надо использовать уникальный код для загрузки, чтоб нормально подтянулись имена колонок.

#### Обработка Null и косметика

In [13]:
feats_bm = datasets['Bank Marketing']['X']
target_bm = datasets['Bank Marketing']['y']
feats_bm.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown


Надо переименовать колонки в DataFrame

In [14]:
proper_names = [
    'age',           # V1
    'job',           # V2
    'marital',       # V3
    'education',     # V4
    'default',       # V5
    'balance',       # V6
    'housing',       # V7
    'loan',          # V8
    'contact',       # V9
    'day_of_week',   # V10
    'month',         # V11
    'duration',      # V12
    'campaign',      # V13
    'pdays',         # V14
    'previous',      # V15
    'poutcome'       # V16
]

feats_bm.columns = proper_names

In [15]:
feats_bm.dtypes

,0
age,int64
job,category
marital,category
education,category
default,category
balance,int64
housing,category
loan,category
contact,category
day_of_week,int64


Nullов в датасете нет. Но надо убрать duration (есть инфо об этом в описании датасета) и скорректировать pdays.

In [16]:
feats_bm = feats_bm.drop('duration', axis=1)
feats_bm['was_contacted'] = (feats_bm['pdays'] != -1).astype(int)
feats_bm.loc[feats_bm['pdays'] == -1, 'pdays'] = 999

Преобразуем таргет в int, а также меняем призаки yes/no на 0 и 1.

In [17]:
binary_cols = ['default', 'housing', 'loan']

for col in binary_cols:
    feats_bm[col] = (feats_bm[col] == 'yes').astype(int)

target_bm = target_bm.map({'1': 0, '2': 1})

target_bm = target_bm.astype(int)

In [18]:
for col in feats_bm.columns:
    print(feats_bm[col].value_counts())

age
32    2085
31    1996
33    1972
34    1930
35    1894
      ... 
95       2
93       2
92       2
88       2
94       1
Name: count, Length: 77, dtype: int64
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64
default
0    44396
1      815
Name: count, dtype: int64
balance
0        3514
1         195
2         156
4         139
3         134
         ... 
14204       1
8205        1
9710        1
7038        1
4416        1
Name: count, Length: 7168, dtype: int64
housing
1    25130
0    20081
Name: count, dtype: int64
loan
0    37967
1     7244
Na

In [19]:
df_bm = feats_bm.copy()
df_bm['target'] = target_bm

output_dir = 'prepared_datasets'
os.makedirs(output_dir, exist_ok=True)

filename = os.path.join(output_dir, 'bank_marketing_cleaned_new.csv')
df_bm.to_csv(filename, index=False)

### 3. California Housing

**Описание:** В данном датасете необходимо предсказать стоимость дома по численным параметрам.

**Ссылка с описанием:** (https://www.kaggle.com/datasets/camnugent/california-housing-prices/data)

In [20]:
url = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'

# Загружаем датасет
df_cal = pd.read_csv(url)
feats_cal = df_cal.drop('median_house_value', axis=1)
target_cal = df_cal['median_house_value']

In [21]:
feats_cal.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY


#### Обработка Null и Косметика

In [22]:
feats_cal.isna().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
ocean_proximity,0


Меняем null в total_bedrooms на медиану.

In [23]:
median_bedrooms = feats_cal['total_bedrooms'].median()

# Заполняем пропуски рассчитанным значением
feats_cal['total_bedrooms'] = feats_cal['total_bedrooms'].fillna(median_bedrooms)

# Меняем тип Ocean_proximity
feats_cal['ocean_proximity'] = feats_cal['ocean_proximity'].astype('category')

#### Анализ на выбросы

In [24]:
numeric_cols = feats_cal.select_dtypes(include = ['number']).columns
for col in numeric_cols:
    q1 = feats_cal[col].quantile(0.25)
    q3 = feats_cal[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = ((feats_cal[col] < lower_bound) | (feats_cal[col] > upper_bound)).sum()

    print(f"{col:25s} | max={feats_cal[col].max():10.2f} | 99%={feats_cal[col].quantile(0.99):10.2f} | выбросов: {outliers:5d} ({outliers/len(feats_cal):.1%})")

longitude                 | max=   -114.31 | 99%=   -116.29 | выбросов:     0 (0.0%)
latitude                  | max=     41.95 | 99%=     40.63 | выбросов:     0 (0.0%)
housing_median_age        | max=     52.00 | 99%=     52.00 | выбросов:     0 (0.0%)
total_rooms               | max=  39320.00 | 99%=  11212.11 | выбросов:  1287 (6.2%)
total_bedrooms            | max=   6445.00 | 99%=   2216.27 | выбросов:  1306 (6.3%)
population                | max=  35682.00 | 99%=   5805.83 | выбросов:  1196 (5.8%)
households                | max=   6082.00 | 99%=   1982.66 | выбросов:  1220 (5.9%)
median_income             | max=     15.00 | 99%=     10.60 | выбросов:   681 (3.3%)


Найденные выбросы по сути выбросами не являются. Это просто и правда очень густонаселенные кварталы. Поэтому убирать или ограничивать их не будем. Единственное - добавим доп.столбцы для инфо по каждому дому.

In [25]:
feats_cal['rooms_per_household'] = feats_cal['total_rooms'] / feats_cal['households']
feats_cal['bedrooms_per_room'] = feats_cal['total_bedrooms'] / feats_cal['total_rooms']
feats_cal['population_per_household'] = feats_cal['population'] / feats_cal['households']

In [26]:
feats_cal.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY,6.281853,0.172096,2.181467


In [27]:
need_cols = ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']
for col in need_cols:
    q1 = feats_cal[col].quantile(0.25)
    q3 = feats_cal[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = ((feats_cal[col] < lower_bound) | (feats_cal[col] > upper_bound)).sum()

    print(f"{col:25s} | max={feats_cal[col].max():10.2f} | 99%={feats_cal[col].quantile(0.99):10.2f} | выбросов: {outliers:5d} ({outliers/len(feats_cal):.1%})")

rooms_per_household       | max=    141.91 | 99%=     10.36 | выбросов:   511 (2.5%)
bedrooms_per_room         | max=      2.82 | 99%=      0.42 | выбросов:   635 (3.1%)
population_per_household  | max=   1243.33 | 99%=      5.39 | выбросов:   711 (3.4%)


В записях по хозяйствам тоже есть выбросы. Но это нормально и связано с географическими особенностями некоторых кварталов. Поэтому выбросы никак не преобразуем.

In [29]:
df_cal = feats_cal.copy()
df_cal['target'] = target_cal

output_dir = 'prepared_datasets'
os.makedirs(output_dir, exist_ok=True)

filename = os.path.join(output_dir, 'california_housing_cleaned_new.csv')
df_cal.to_csv(filename, index=False)

### 4. Superconductivity

**Описание:** В данном датасете необходимо предсказать критическую температуру перехода материала в сверхпроводящее состояние по другим его характеристикам.

**Ссылка с описанием:** (https://archive.ics.uci.edu/dataset/464/superconductivty+data)

In [31]:
import openml
import pandas as pd

# Загружаем датасет Superconductivity по ID
dataset = openml.datasets.get_dataset(43174)

# Получаем данные с правильными названиями колонок
X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute
)

feats_sc = pd.DataFrame(X, columns=attribute_names)
target_sc = pd.Series(y, name=dataset.default_target_attribute)

In [32]:
not_num_cols = feats_sc.select_dtypes(exclude = ['number']).columns

Здесь нет Nullов, сразу кидаем в CSV.

In [35]:
df_sc = feats_sc.copy()
df_sc['target'] = target_sc

output_dir = 'prepared_datasets'
os.makedirs(output_dir, exist_ok=True)

filename = os.path.join(output_dir, 'superconductivity_cleaned_new.csv')
df_sc.to_csv(filename, index=False)

### 5. Spambase (определение спама)



**Описание:** В данном датасете необходимо определить письма со спамом. Датасет состоит их электорнных писем и имеет 57 непрерывных признаков со статистиками по словам и символам.

**Ссылка с описанием:** (https://archive-beta.ics.uci.edu/dataset/94/spambase)

In [36]:
feats_sp = datasets['Spambase']['X']
target_sp = datasets['Spambase']['y']
feats_sp.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,word_freq_conference,char_freq_%3B,char_freq_%28,char_freq_%5B,char_freq_%21,char_freq_%24,char_freq_%23,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191


Здесь также нет Nullов. Есть выбросы, но они имеют физическую природу и несут информацию, поэтому с ними разбираемся уже на препроцессинге в пайплайне.

In [37]:
target_sp = target_sp.astype('int')

In [38]:
df_sp = feats_sp.copy()
df_sp['target'] = target_sp

output_dir = 'prepared_datasets'
os.makedirs(output_dir, exist_ok=True)

filename = os.path.join(output_dir, 'spambase_cleaned_new.csv')
df_sp.to_csv(filename, index=False)

###